In [2]:
!pip install -q monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 34.0 MB/s eta 0:00:00


In [ ]:
"""
evaluate.py (simple version)
=============================
Loads the trained model and computes Dice, IoU, HD95, Precision, and Recall
on the validation set. Kept short and linear on purpose so it's easy to
walk through step by step.
"""

import os, json
import numpy as np
import torch

from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRangePercentilesd, NormalizeIntensityd, CropForegroundd,
    SpatialPadd, AsDiscrete,
)
from monai.data import Dataset, DataLoader
from monai.networks.nets import UNet
from monai.metrics import DiceMetric, MeanIoU, HausdorffDistanceMetric
from monai.metrics.confusion_matrix import get_confusion_matrix
from monai.inferers import sliding_window_inference

# ---------------------------------------------------------------------
# 1. Paths and settings (must match the training notebook)
# ---------------------------------------------------------------------
PROJECT_DIR = "/content/drive/MyDrive/Monai_ACDC-Project-work"
CHECKPOINT_PATH = os.path.join(PROJECT_DIR, "checkpoints", "best_model.pt")
VAL_SPLIT_PATH = os.path.join(PROJECT_DIR, "splits", "val_files.json")

LABEL_MAP = {0: "background", 1: "RV", 2: "MYO", 3: "LV"}
CLASSES = ["RV", "MYO", "LV"]          # foreground classes only
NUM_CLASSES = 4
PATCH_SIZE = (128, 128, 16)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------------------
# 2. Load validation file list (created earlier by the training notebook)
# ---------------------------------------------------------------------
with open(VAL_SPLIT_PATH) as f:
    val_files = json.load(f)
print(f"Validation cases: {len(val_files)}")

# ---------------------------------------------------------------------
# 3. Same preprocessing used for validation during training (no augmentation)
# ---------------------------------------------------------------------
val_transforms = Compose([
    LoadImaged(keys=("image", "label")),
    EnsureChannelFirstd(keys=("image", "label")),
    Orientationd(keys=("image", "label"), axcodes="RAS"),
    Spacingd(keys=("image", "label"), pixdim=(1.5, 1.5, 3.0), mode=("bilinear", "nearest")),
    ScaleIntensityRangePercentilesd(keys=["image"], lower=0.5, upper=99.5, b_min=0.0, b_max=1.0, clip=True),
    NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
    CropForegroundd(keys=("image", "label"), source_key="image", margin=15, allow_smaller=True),
    SpatialPadd(keys=("image", "label"), spatial_size=PATCH_SIZE),
])

val_loader = DataLoader(Dataset(data=val_files, transform=val_transforms), batch_size=1)

# ---------------------------------------------------------------------
# 4. Load the trained model
# ---------------------------------------------------------------------
from monai.networks.layers import Norm

model = UNet(
    spatial_dims=3, in_channels=1, out_channels=NUM_CLASSES,
    channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2), num_res_units=2,
    norm=Norm.BATCH, dropout=0.1,   # must match training exactly, or the checkpoint won't load
).to(DEVICE)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Loaded trained model from epoch", checkpoint["epoch"])

# ---------------------------------------------------------------------
# 5. Metrics
# ---------------------------------------------------------------------
post_pred = AsDiscrete(argmax=True, to_onehot=NUM_CLASSES)
post_label = AsDiscrete(to_onehot=NUM_CLASSES)

dice_metric = DiceMetric(include_background=False, reduction="none")
iou_metric = MeanIoU(include_background=False, reduction="none")
hd95_metric = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="none")

def precision_recall(pred, label):
    # get_confusion_matrix gives [TP, FP, TN, FN] per class
    tp, fp, tn, fn = get_confusion_matrix(pred, label, include_background=False).unbind(-1)
    precision = tp / (tp + fp + 1e-7)
    recall = tp / (tp + fn + 1e-7)
    return precision.cpu().numpy()[0], recall.cpu().numpy()[0]

# ---------------------------------------------------------------------
# 6. Run inference + compute metrics for every validation case
# ---------------------------------------------------------------------
results = {cls: {"dice": [], "iou": [], "hd95": [], "precision": [], "recall": []} for cls in CLASSES}

with torch.no_grad():
    for batch in val_loader:
        image = batch["image"].to(DEVICE)
        label = batch["label"].to(DEVICE)

        output = sliding_window_inference(image, roi_size=PATCH_SIZE, sw_batch_size=1, predictor=model)

        pred_oh = [post_pred(x) for x in output]
        label_oh = [post_label(x) for x in label]
        pred_stack, label_stack = torch.stack(pred_oh), torch.stack(label_oh)

        dice = dice_metric(pred_oh, label_oh).cpu().numpy()[0]
        iou = iou_metric(pred_oh, label_oh).cpu().numpy()[0]
        hd95 = hd95_metric(pred_oh, label_oh).cpu().numpy()[0]
        precision, recall = precision_recall(pred_stack, label_stack)

        for i, cls in enumerate(CLASSES):
            results[cls]["dice"].append(dice[i])
            results[cls]["iou"].append(iou[i])
            results[cls]["hd95"].append(hd95[i])
            results[cls]["precision"].append(precision[i])
            results[cls]["recall"].append(recall[i])

# ---------------------------------------------------------------------
# 7. Print the final scores
# ---------------------------------------------------------------------
print(f"\n{'Class':<8} {'Dice':>8} {'IoU':>8} {'HD95':>8} {'Precision':>10} {'Recall':>8}")
for cls in CLASSES:
    r = results[cls]
    print(f"{cls:<8} {np.mean(r['dice']):8.4f} {np.mean(r['iou']):8.4f} "
          f"{np.mean(r['hd95']):8.2f} {np.mean(r['precision']):10.4f} {np.mean(r['recall']):8.4f}")

Validation cases: 30


/usr/local/lib/python3.13/dist-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


Loaded trained model from epoch 14


/usr/local/lib/python3.13/dist-packages/monai/utils/deprecate_utils.py:220: FutureWarning: monai.metrics.utils get_mask_edges:always_return_as_numpy: Argument `always_return_as_numpy` has been deprecated since version 1.5.0. It will be removed in version 1.7.0. The option is removed and the return type will always be equal to the input type.
  warn_deprecated(argname, msg, warning_category)



Class        Dice      IoU     HD95  Precision   Recall
RV         0.5339   0.3872    66.31     0.5882   0.5611
MYO        0.5696   0.4063    50.03     0.5015   0.6902
LV         0.8162   0.6951    29.92     0.8011   0.8531
